In [1]:
print('hello world')

hello world


In [5]:
import pandas as pd
import numpy as np
import re
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
match_df = pd.read_csv('../dataset/matching/job_resume_fit.csv')

In [4]:
print("Dataset shape:", match_df.shape)
print("\nColumns:")
print(match_df.columns)
print("\nFirst 5 rows:")
print(match_df.head())

Dataset shape: (2385, 10)

Columns:
Index(['ID', 'resume_text', 'job_text', 'category', 'job_required_skills',
       'resume_skill_list', 'ai_matched_skills', 'ai_match_score',
       'skill_string_match_score', 'fuzzy_match_score'],
      dtype='str')

First 5 rows:
         ID                                        resume_text  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                            job_text category  \
0  Human Resources Business Partner\n------------...       HR   
1  Human Resources Business Partner\n------------...       HR   
2  Human Resources Business Partner\n------------...       HR   
3  Human Resources Business Partner\n------------...       HR   
4  Human 

In [7]:
match_df = match_df[[
    'resume_text',
    'job_text',
    'category',
    'job_required_skills',
    'resume_skill_list',
    'ai_matched_skills',
    'ai_match_score',
]].copy()

match_df.dropna(inplace=True)

print("Shape after cleaning: ", match_df.shape)
print(match_df.head())

Shape after cleaning:  (2385, 7)
                                         resume_text  \
0           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1           HR SPECIALIST, US HR OPERATIONS      ...   
2           HR DIRECTOR       Summary      Over 2...   
3           HR SPECIALIST       Summary    Dedica...   
4           HR MANAGER         Skill Highlights  ...   

                                            job_text category  \
0  Human Resources Business Partner\n------------...       HR   
1  Human Resources Business Partner\n------------...       HR   
2  Human Resources Business Partner\n------------...       HR   
3  Human Resources Business Partner\n------------...       HR   
4  Human Resources Business Partner\n------------...       HR   

                                 job_required_skills  \
0  ['human resources management', 'talent acquisi...   
1  ['human resources management', 'talent acquisi...   
2  ['human resources management', 'talent acquisi...   
3  ['human reso

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+www\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
match_df['clean_resume'] = match_df['resume_text'].apply(clean_text)
match_df['clean_job'] = match_df['job_text'].apply(clean_text)

print(match_df[['clean_resume', 'clean_job']].head(2))

                                        clean_resume  \
0  hr administrator marketing associate hr admini...   
1  hr specialist us hr operations summary versati...   

                                           clean_job  
0  human resources business partner we are seekin...  
1  human resources business partner we are seekin...  


In [10]:
def safe_literal_eval(value):
    try:
        return ast.literal_eval(value)
    except:
        return []

match_df['job_required_skills'] = match_df['job_required_skills'].apply(safe_literal_eval)
match_df['resume_skill_list'] = match_df['resume_skill_list'].apply(safe_literal_eval)
match_df['ai_match_skills'] = match_df['ai_matched_skills'].apply(safe_literal_eval)

In [11]:
print("Job Skills:", match_df.loc[0, 'job_required_skills'])
print("Resume Skills:", match_df.loc[0, 'resume_skill_list'])
print("AI Matched Skills:", match_df.loc[0, 'ai_match_skills'])

Job Skills: ['human resources management', 'talent acquisition', 'employment law compliance', 'employee relations', 'performance management', 'compensation and benefits', 'HRIS systems', 'HR analytics', 'organizational development', 'change management', 'project management', 'training and development', 'succession planning', 'business acumen', 'data analysis', 'Microsoft Office Suite', 'interpersonal skills', 'communication', 'influencing', 'problem solving', 'critical thinking', 'ethical judgment', 'confidentiality', 'teamwork']
Resume Skills: ['customer service', 'team management', 'marketing', 'conflict resolution', 'training', 'multi-tasking', 'client relations', 'medical billing', 'data analysis', 'icd-9', 'cpt', 'financial management', 'budgeting', 'advertising', 'human resources', 'insurance', 'labor relations', 'performance reviews', 'personnel management', 'public relations', 'website design', 'dedication', 'leadership', 'communication', 'problem solving', 'organizational skil

In [12]:
def calculate_tfidf_similarity(resume_text, job_text):
    texts = [resume_text, job_text]
    
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(texts)
    
    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(similarity * 100, 2)

In [13]:
def calculate_skill_match(job_skills, resume_skills):
    job_set = set([str(skill).lower().strip() for skill in job_skills])
    resume_set = set([str(skill).lower().strip() for skill in resume_skills])
    
    matched_skills = sorted(job_set.intersection(resume_set))
    missing_skills = sorted(job_set - resume_set)
    
    if len(job_set) == 0:
        skill_score = 0
    else:
        skill_score = (len(matched_skills) / len(job_set)) * 100
    
    return round(skill_score, 2), matched_skills, missing_skills

In [14]:
sample_row = match_df.iloc[0]

resume_text = sample_row['clean_resume']
job_text = sample_row['clean_job']
job_skills = sample_row['job_required_skills']
resume_skills = sample_row['resume_skill_list']

tfidf_score = calculate_tfidf_similarity(resume_text, job_text)
skill_score, matched_skills, missing_skills = calculate_skill_match(job_skills, resume_skills)

print("Category:", sample_row['category'])
print("TF-IDF Similarity Score:", tfidf_score)
print("Skill Match Score:", skill_score)
print("Matched Skills:", matched_skills)
print("Missing Skills:", missing_skills)
print("Dataset AI Match Score:", sample_row['ai_match_score'])

Category: HR
TF-IDF Similarity Score: 15.4
Skill Match Score: 12.5
Matched Skills: ['communication', 'data analysis', 'problem solving']
Missing Skills: ['business acumen', 'change management', 'compensation and benefits', 'confidentiality', 'critical thinking', 'employee relations', 'employment law compliance', 'ethical judgment', 'hr analytics', 'hris systems', 'human resources management', 'influencing', 'interpersonal skills', 'microsoft office suite', 'organizational development', 'performance management', 'project management', 'succession planning', 'talent acquisition', 'teamwork', 'training and development']
Dataset AI Match Score: 49.4


In [16]:
def calculate_final_score(tfidf_score, skill_score, skill_weight=0.6, text_weight=0.4):
    final_score = (skill_score * skill_weight) + (tfidf_score * text_weight)
    return round(final_score, 2)

final_score = calculate_final_score(tfidf_score, skill_score)

print("TF-IDF Score:", tfidf_score)
print("Skill Score:", skill_score)
print("Final Match Score:", final_score)

TF-IDF Score: 15.4
Skill Score: 12.5
Final Match Score: 13.66


In [17]:
def match_resume_to_job(resume_text, job_text, job_skills, resume_skills):
    # clean texts
    resume_text_clean = clean_text(resume_text)
    job_text_clean = clean_text(job_text)
    
    # calculate scores
    tfidf_score = calculate_tfidf_similarity(resume_text_clean, job_text_clean)
    skill_score, matched_skills, missing_skills = calculate_skill_match(job_skills, resume_skills)
    final_score = calculate_final_score(tfidf_score, skill_score)
    
    return {
        "tfidf_score": tfidf_score,
        "skill_score": skill_score,
        "final_score": final_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills
    }

In [18]:
sample_row = match_df.iloc[0]

result = match_resume_to_job(
    resume_text=sample_row['resume_text'],
    job_text=sample_row['job_text'],
    job_skills=sample_row['job_required_skills'],
    resume_skills=sample_row['resume_skill_list']
)

print(result)

{'tfidf_score': np.float64(15.4), 'skill_score': 12.5, 'final_score': np.float64(13.66), 'matched_skills': ['communication', 'data analysis', 'problem solving'], 'missing_skills': ['business acumen', 'change management', 'compensation and benefits', 'confidentiality', 'critical thinking', 'employee relations', 'employment law compliance', 'ethical judgment', 'hr analytics', 'hris systems', 'human resources management', 'influencing', 'interpersonal skills', 'microsoft office suite', 'organizational development', 'performance management', 'project management', 'succession planning', 'talent acquisition', 'teamwork', 'training and development']}


In [19]:
results = []

for i in range(10):
    row = match_df.iloc[i]
    
    result = match_resume_to_job(
        resume_text=row['resume_text'],
        job_text=row['job_text'],
        job_skills=row['job_required_skills'],
        resume_skills=row['resume_skill_list']
    )
    
    results.append({
        "category": row['category'],
        "our_score": result['final_score'],
        "dataset_ai_score": row['ai_match_score'],
        "matched_skills_count": len(result['matched_skills']),
        "missing_skills_count": len(result['missing_skills'])
    })

comparison_df = pd.DataFrame(results)
comparison_df

,category,our_score,dataset_ai_score,matched_skills_count,missing_skills_count
0,HR,13.66,49.40,3,21
1,HR,15.54,77.64,4,20
2,HR,22.49,100.00,5,19
3,HR,9.28,32.63,2,22
4,HR,33.39,86.03,7,17
5,HR,16.15,73.15,4,20
6,HR,25.81,71.36,7,17
7,HR,13.73,86.03,2,22
8,HR,23.74,95.81,6,18
9,HR,32.25,90.92,8,16
